In [1]:
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
from seqeval.metrics import classification_report, f1_score

In [2]:
from datasets import load_dataset

dataset = load_dataset("conll2003")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/datasets/load.py:1486: FutureWarning: The repository for conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2003
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


In [3]:
label_list = dataset["train"].features["pos_tags"].feature.names

id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

print(label_list)

['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']


In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [5]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["pos_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [6]:
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

In [7]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [9]:
import numpy as np
from seqeval.metrics import f1_score

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    return {
        "f1": f1_score(true_labels, true_predictions)
    }

In [10]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [11]:
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    print(classification_report(true_labels, true_predictions))  # ✅ important

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }

In [12]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [13]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.758182,0.263875,0.908402,0.910275,0.909338
2,0.211714,0.240573,0.917058,0.915055,0.916055


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: : seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: . seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarni

              precision    recall  f1-score   support

           '       0.00      0.00      0.00        11
           B       0.87      0.84      0.86      1907
          BD       0.94      0.96      0.95      2224
          BG       0.87      0.90      0.89       699
          BN       0.86      0.83      0.85       928
          BP       0.87      0.81      0.84       365
          BR       0.00      0.00      0.00        53
          BS       0.00      0.00      0.00        18
          BZ       0.92      0.91      0.92       509
           C       1.00      1.00      1.00       932
           D       0.92      0.97      0.94      3231
          DT       0.94      0.88      0.91       162
           H       0.00      0.00      0.00         5
           J       0.83      0.81      0.82      2765
          JR       0.66      0.92      0.77       105
          JS       0.74      0.82      0.78        78
           N       0.90      0.88      0.89      8147
          NP       0.81    

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: : seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: . seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarni

              precision    recall  f1-score   support

           '       1.00      0.36      0.53        11
           B       0.91      0.85      0.88      1907
          BD       0.94      0.96      0.95      2224
          BG       0.90      0.89      0.89       699
          BN       0.88      0.84      0.86       928
          BP       0.85      0.87      0.86       365
          BR       0.88      0.28      0.43        53
          BS       0.00      0.00      0.00        18
          BZ       0.92      0.92      0.92       509
           C       1.00      1.00      1.00       932
           D       0.92      0.97      0.95      3231
          DT       0.94      0.88      0.91       162
           H       0.00      0.00      0.00         5
           J       0.85      0.82      0.84      2765
          JR       0.75      0.94      0.84       105
          JS       0.74      0.85      0.79        78
           N       0.91      0.89      0.90      8147
          NP       0.83    

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1756, training_loss=0.377254609909579, metrics={'train_runtime': 193.2461, 'train_samples_per_second': 145.317, 'train_steps_per_second': 9.087, 'total_flos': 340710744969678.0, 'train_loss': 0.377254609909579, 'epoch': 2.0})

In [18]:
def predict_sentence(sentence):
    tokens = sentence.split()
    tokenized_inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)

    # Move inputs to the same device as the model
    inputs_on_device = {k: v.to(model.device) for k, v in tokenized_inputs.items()}

    outputs = model(**inputs_on_device)
    predictions = np.argmax(outputs.logits.detach().cpu().numpy(), axis=2)

    word_ids = tokenized_inputs.word_ids()
    predicted_labels = []

    for idx, word_idx in enumerate(word_ids):
        if word_idx is not None:
            predicted_labels.append(label_list[predictions[0][idx]])

    print("\nSentence Prediction:\n")
    for word, label in zip(tokens, predicted_labels):
        print(f"{word} → {label}")

predict_sentence("John works at Google in California")


Sentence Prediction:

John → NNP
works → VBZ
at → IN
Google → NNP
in → IN
California → NNP


##  Comparison between POS Tagging and Chunking

### 1. POS Tagging (Part-of-Speech Tagging)
- POS tagging assigns grammatical labels to each word in a sentence.
- It is a **word-level task**.
- Examples of POS tags include:
  - NN (Noun)
  - VB (Verb)
  - JJ (Adjective)

**Example:**
Sentence: "John works at Google"

| Word   | POS Tag |
|--------|--------|
| John   | NNP    |
| works  | VBZ    |
| at     | IN     |
| Google | NNP    |

---

### 2. Chunking (Phrase Detection)
- Chunking groups words into meaningful phrases.
- It is a **phrase-level task**.
- Examples of chunk tags include:
  - NP (Noun Phrase)
  - VP (Verb Phrase)

**Example:**
Sentence: "John works at Google"

| Word   | Chunk Tag |
|--------|----------|
| John   | B-NP     |
| works  | B-VP     |
| at     | B-PP     |
| Google | B-NP     |

---

### 3. Key Differences

| Feature            | POS Tagging              | Chunking                  |
|-------------------|--------------------------|---------------------------|
| Level             | Word-level               | Phrase-level              |
| Output            | Grammar labels           | Phrase groups             |
| Complexity        | Easier                   | More complex              |
| Example Output    | NN, VB, JJ               | NP, VP, PP                |

---

### 4. Conclusion
- POS tagging is simpler and focuses on identifying the grammatical role of each word.
- Chunking builds on POS tagging and identifies higher-level structures like phrases.
- Therefore, chunking is generally more complex than POS tagging.

## Report / Blog

### 1. Introduction
In this assignment, a transformer-based model (DistilBERT) was fine-tuned to perform Part-of-Speech (POS) tagging using token classification techniques. The CoNLL-2003 dataset was used for training and evaluation.

---

### 2. Approach
- The dataset was loaded using the Hugging Face `datasets` library.
- Tokenization was performed using the DistilBERT tokenizer.
- Labels were aligned with tokens, handling subwords and special tokens using `-100`.
- The model was trained using the Hugging Face `Trainer` API.
- Evaluation was performed using the `seqeval` library.

---

### 3. Challenges Faced
- **Token-Label Alignment:** Handling subword tokenization required careful alignment of labels.
- **Special Tokens Handling:** Tokens like `[CLS]` and `[SEP]` needed to be ignored using `-100`.
- **Device Issues:** Ensuring tensors were on the same device (CPU/GPU) during inference.
- **Evaluation Warnings:** The `seqeval` library is designed for NER, so warnings appeared when used for POS tagging.

---

### 4. Observations
- Transformer models like DistilBERT perform very well on sequence labeling tasks.
- The model achieved strong performance with an approximate **F1 score of ~0.92**.
- Subword tokenization improves model understanding but adds preprocessing complexity.
- The Hugging Face Trainer simplifies training and evaluation significantly.

---

### 5. Results
- Precision: ~0.92  
- Recall: ~0.92  
- F1 Score: ~0.92  

These results indicate that the model performs well in predicting POS tags.

---

### 6. Conclusion
This assignment demonstrated how transformer models can be effectively used for NLP tasks like POS tagging. It also highlighted important challenges such as token alignment and evaluation handling. Overall, the model showed strong performance and practical usability.

---

### 7. Future Improvements
- Extend the model to perform chunking as well.
- Use larger datasets for better generalization.
- Experiment with different transformer models like BERT or RoBERTa.